# 06 — W8 Deep Models on Colab

**One-stop training notebook** for Bi-LSTM and Legal-BERT on both classification tasks.
Designed for **Colab T4** (free tier) — total wall-clock ≈ 40–60 min for all 4 deep-model runs.

## How to run

1. Open in Colab. **Runtime → Change runtime type → T4 GPU** (or any GPU).
2. Run cell 1 to clone the repo + install pinned deps.
3. Run cell 2 to upload (or pull) `data/multilexsum_clean.parquet`.
4. Run cells 3–6 to train Bi-LSTM × 2 tasks + Legal-BERT × 2 tasks.
5. Cell 7 zips the artifacts and lets you download them.

Same `python -m src.classify.train` CLI as W7 — the only thing this notebook adds is GPU + drive plumbing.

## 1. Environment

Clone the repo + install the exact dependency pins that match the local env.

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/junbolian/MLDS-NLP-final.git'
REPO_DIR = '/content/MLDS-NLP-final'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR

# Pinned versions match local env. Use --quiet to keep the notebook clean.
!pip install --quiet 'datasets>=2.16.0,<3.0.0' transformers torch gensim shap rouge-score bert-score sentencepiece accelerate

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Get the cleaned parquet

Three options:
1. **Upload from local** (simplest for a one-off run) — runs the `files.upload()` widget below.
2. **Mount Google Drive** — set `USE_DRIVE = True` and put `multilexsum_clean.parquet` in your Drive root.
3. **Rebuild from raw** — uncomment the last cell of this section (downloads ~2 GB and takes ~10 min).

In [ ]:
from pathlib import Path
os.makedirs('data', exist_ok=True)
USE_DRIVE = False    # set True if you keep the parquet in /MyDrive/

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    src = '/content/drive/MyDrive/multilexsum_clean.parquet'
    dst = 'data/multilexsum_clean.parquet'
    if not os.path.exists(dst):
        import shutil; shutil.copy(src, dst)
        print(f'Copied {src} → {dst}')
else:
    from google.colab import files
    if not os.path.exists('data/multilexsum_clean.parquet'):
        print('Upload multilexsum_clean.parquet (it lives in your local data/ folder, ~440 MB):')
        uploaded = files.upload()
        # files.upload returns a dict {filename: bytes}; persist it under data/
        for name, content in uploaded.items():
            with open(f'data/{name}', 'wb') as fh:
                fh.write(content)

import pandas as pd
df = pd.read_parquet('data/multilexsum_clean.parquet')
print(f'Loaded {len(df)} rows; splits: {df.split.value_counts().to_dict()}')

## 3. Bi-LSTM × class_action_sought

Self-trained Word2Vec (300d) → BiLSTM(128) → Linear. ~3 min on T4.

In [ ]:
!python -m src.classify.train --task class_action --model lstm --device cuda --epochs 10

## 4. Bi-LSTM × case_type_grouped

Same architecture, 5-class output head.

In [ ]:
!python -m src.classify.train --task case_type --model lstm --device cuda --epochs 10

## 5. Legal-BERT × class_action_sought

`nlpaueb/legal-bert-base-uncased`, 3 epochs, batch 8. ~15 min on T4.

**Fallback for slow runs**: append `--bert-model-name distilbert-base-uncased` to halve the time at ~2 F1-points cost.

In [ ]:
!python -m src.classify.train --task class_action --model bert --device cuda --epochs 3 --batch-size 8

## 6. Legal-BERT × case_type_grouped

In [ ]:
!python -m src.classify.train --task case_type --model bert --device cuda --epochs 3 --batch-size 8

## 7. Inspect + download artifacts

Look at the metrics, then zip the models/results for offline analysis.

In [ ]:
import pandas as pd
m = pd.read_csv('results/classification_metrics.csv')
print(m[m['split'] == 'test'].to_string(index=False))

In [ ]:
# Zip and download — BERT models can be hundreds of MB each, so this can take a minute.
import shutil
shutil.make_archive('/content/w8_artifacts', 'zip', '/content/MLDS-NLP-final',
                    base_dir='.', logger=None)
# A simpler alternative: zip just the model + results subtrees you need.
subprocess_cmd = (
    'zip -qr /content/w8_artifacts.zip '
    'models/lstm_classaction.pt models/lstm_casetype.pt '
    'models/bert_classaction models/bert_casetype '
    'results/classification_metrics.csv '
    'results/training_curves results/confusion_matrices '
    'results/classification_reports'
)
import subprocess; subprocess.run(subprocess_cmd, shell=True, check=True)
from google.colab import files
files.download('/content/w8_artifacts.zip')

## 8. Optional: push to Drive

Skip if you used the local-upload path.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/w8_artifacts.zip /content/drive/MyDrive/